# Inventory Optimization

## 📊 Business Context
Forecast stock levels.

**Analytical Approach:** Forecasting
This notebook utilizes advanced analytics to derive actionable insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Data Generation
def generate_timeseries(n=365):
    dates = pd.date_range(start='2023-01-01', periods=n, freq='D')
    t = np.arange(n)
    # Trend + Seasonality + Noise
    trend = 0.5 * t
    season = 10 * np.sin(2 * np.pi * t / 30) # Monthly seasonality
    noise = np.random.normal(0, 5, n)
    values = 100 + trend + season + noise
    
    return pd.DataFrame({'Date': dates, 'Stock_Level': values}).set_index('Date')

df = generate_timeseries(730)
df.plot(figsize=(12,6), title='Historical Data')
plt.show()

In [ ]:
# Forecasting Engine
class Forecaster:
    def __init__(self, df):
        self.df = df
        self.model = None
        
    def train_predict(self, days=30):
        # Train/Test Split
        train = self.df.iloc[:-days]
        test = self.df.iloc[-days:]
        
        # Holt-Winters Exponential Smoothing
        self.model = ExponentialSmoothing(train, seasonal='add', seasonal_periods=30).fit()
        preds = self.model.forecast(days)
        
        # Evaluation
        mae = mean_absolute_error(test, preds)
        print(f'MAE: {mae:.2f}')
        
        # Plot
        plt.figure(figsize=(12,6))
        plt.plot(train.index, train, label='Train')
        plt.plot(test.index, test, label='Test')
        plt.plot(test.index, preds, label='Forecast', linestyle='--')
        plt.legend()
        plt.title('Forecast vs Actuals')
        plt.show()
        return preds

forecaster = Forecaster(df)
forecast = forecaster.train_predict(30)